# EXPT NO: 10 – BUILDING A SIMPLE LLM CHATBOT USING GRADIO

**Student Roll No:** 24BAD108  


## PART A – ENVIRONMENT SETUP AND MODEL SELECTION


In [1]:
# Cell 1: Install the required libraries
%pip install -q -U "transformers>=4.45.0" accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.8 MB/s eta 0:00:00


In [2]:
# Cell 2: Import libraries and check hardware
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU; generation may be slower.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
# Cell 3: Load the pretrained model and tokenizer
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
)
model.eval()

print("Model loaded:", MODEL_ID)
print("Model device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Model device: cuda:0


## PART B – PROMPT HANDLING AND RESPONSE GENERATION


In [4]:
# Cell 4: Define a response-generation function
SYSTEM_PROMPT = "You are a helpful, polite assistant. Answer clearly and concisely. If unsure, say so."

def generate_response(message, history=None, max_new_tokens=128):
    if message is None or not str(message).strip():
        return "Please enter a message."

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in (history or [])[-8:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in ("user", "assistant") and isinstance(content, str) and content.strip():
                messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": str(message).strip()})

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return response or "I could not generate a response. Please try another prompt."
    except Exception as exc:
        return f"Error while generating the response: {type(exc).__name__}: {exc}"

print("Response function defined.")

Response function defined.


In [5]:
# Cell 5: Test one simple prompt
prompt = "What is artificial intelligence?"
print("Prompt:", prompt)
print("Response:", generate_response(prompt))

Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and act like humans. It encompasses a wide range of technologies used by computers to perform tasks that typically require human intelligence, such as decision-making, problem-solving, learning from experience, and understanding natural language. AI involves creating software, hardware, and algorithms that can process information, learn from data, make decisions, and interact with users in ways that mimic human behavior and cognition.


In [6]:
# Cell 6: Test five different prompts
prompts = [
    "Explain machine learning in simple words.",
    "Give three tips for preparing for an exam.",
    "Write a short Python program to add two numbers.",
    "What is the difference between AI and machine learning?",
    "Give a two-sentence summary of renewable energy."
]

for i, prompt in enumerate(prompts, start=1):
    print(f"\nTEST {i}: {prompt}")
    print(generate_response(prompt, history=[], max_new_tokens=80))
    print("-" * 55)


TEST 1: Explain machine learning in simple words.
Machine learning is a type of AI that allows computers to learn from data and improve their performance on specific tasks without being explicitly programmed. It uses algorithms to identify patterns and make predictions based on input data. Machine learning involves building models or systems that can learn from data and adapt to new information, often used for tasks like image recognition, natural language processing, and predictive analytics.
-------------------------------------------------------

TEST 2: Give three tips for preparing for an exam.
1. Create a study schedule: Allocate specific times each day to review your notes, practice problems, and review material from previous exams or classes.

2. Practice regularly: Set aside time every day or week to work on practice tests or assignments. This will help you familiarize yourself with the format of the exam and improve your ability to perform under pressure.

3. Stay organized:

## PART C – BUILDING THE CHATBOT INTERFACE


In [7]:
# Cell 7: Create the Gradio chatbot

def chatbot_fn(message, history):
    return generate_response(message, history=history)

demo = gr.ChatInterface(
    fn=chatbot_fn,
    title="Simple LLM Chatbot",
    description="A chatbot powered by a pretrained Transformer model.",
    examples=[
        "Explain neural networks in simple words.",
        "Give me a study timetable for one day.",
        "Write a Python function to check an even number.",
        "What is climate change?"
    ],
    save_history=True
)

print("Gradio interface created.")

Gradio interface created.


In [8]:
# Cell 8: Launch the chatbot in Google Colab
# share=True provides a temporary link for demonstration.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f386a17a86d36b3b09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## PART D – TESTING AND DEPLOYMENT


In [9]:
# Cell 9: Basic empty-prompt and conversation-history checks
print("Empty prompt test:")
print(generate_response("   "))

sample_history = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Nice to meet you, Alex."}
]
print("\nConversation-context test:")
print(generate_response("What is my name?", history=sample_history, max_new_tokens=60))

Empty prompt test:
Please enter a message.

Conversation-context test:
Your name is Alex.


## Create Files for Local Execution


In [10]:
# Cell 10: Write app.py and requirements.txt
app_source = r'''import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT = "You are a helpful, polite assistant. Answer clearly and concisely."
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto")
model.eval()

def chatbot_fn(message, history):
    if message is None or not str(message).strip():
        return "Please enter a message."
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in (history or [])[-8:]:
        if isinstance(item, dict):
            role, content = item.get("role"), item.get("content", "")
            if role in ("user", "assistant") and isinstance(content, str) and content.strip():
                messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": str(message).strip()})
    try:
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128, do_sample=True,
                                     temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                                     pad_token_id=tokenizer.eos_token_id)
        new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return answer or "I could not generate a response. Please try again."
    except Exception as exc:
        return f"Error: {type(exc).__name__}: {exc}"

demo = gr.ChatInterface(fn=chatbot_fn, title="Simple LLM Chatbot", save_history=True)
if __name__ == "__main__":
    demo.launch()
'''

requirements = "torch\ntransformers>=4.45.0\naccelerate\ngradio\n"
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_source.strip() + "\n")
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements.replace("\\n", "\n"))
print("Created app.py and requirements.txt")

Created app.py and requirements.txt
